# Context-Window Degradation — Colab Experiment Runner

Follow the cells from top to bottom. Use **Runtime → Change runtime type → GPU** before starting. A Tesla T4 is suitable when Colab provides one, but another available GPU may also work.

**Research integrity note:** the benchmark includes a controlled synthetic contradiction. Study B's inserted result is a synthetic counterfactual, not a published biomedical finding. The notebook does not contain pre-filled experimental results; the CSV files are created only after a successful run.

## 0. Start or update the repository

Run this cell first. It safely updates an existing checkout or clones the branch if it is not present.

In [ ]:
%cd /content
from pathlib import Path
import subprocess, sys

REPO_DIR = Path('/content/AITR_Lab02_PSE2026002_ContextDegradation')
REPO_URL = 'https://github.com/HarshKumarSinghCU/AITR_Lab02_PSE2026002_ContextDegradation.git'
BRANCH = 'feat/reproducible-experiment-pipeline'

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)

%cd /content/AITR_Lab02_PSE2026002_ContextDegradation
print('Repository ready:', Path.cwd())
print('Branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())

# Install the repository's declared dependencies in the active Colab runtime.
%pip install -q -r requirements.txt
print('Dependency installation completed.')

## 1. Check the runtime and run the tests

If the tests fail, stop here and share the first error. Do not continue to the experiment until the validation cell passes.

In [ ]:
import torch, subprocess, sys
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU not detected. Change runtime type to GPU before running the full experiment.')

subprocess.run([sys.executable, 'scripts/env_sanity_check.py'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
print('Environment check and unit tests completed successfully.')

## 2. Run the five-cluster pilot (135 records)

This is the compute-heavy step. On the first run, the code downloads PubMedQA and model weights. Keep the Colab tab/runtime active and wait for the cell to finish. Do not run the plotting cell while this cell is running.

If this cell raises an exception, the notebook will stop here and show the original error instead of producing a misleading missing-CSV error.

In [ ]:
from pathlib import Path
import subprocess, sys

raw_csv = Path('results/experiment_results.csv')
summary_csv = Path('results/summary_metrics.csv')
metadata_json = Path('results/run_metadata.json')

# Remove stale outputs so an unsuccessful rerun cannot be mistaken for a fresh result.
for output_file in (raw_csv, summary_csv, metadata_json):
    if output_file.exists():
        output_file.unlink()

subprocess.run([sys.executable, 'scripts/run_factorial_experiment.py', '--clusters', '5'], check=True)

missing = [str(p) for p in (raw_csv, summary_csv, metadata_json) if not p.is_file()]
if missing:
    raise RuntimeError('Experiment process ended but expected output files are missing: ' + ', '.join(missing))

import json
run_info = json.loads(metadata_json.read_text())
print('\nPilot run metadata:', run_info)
print('Raw result rows:', sum(1 for _ in raw_csv.open()) - 1)
print('Pilot outputs verified.')

## 3. Generate figures and inspect the measured summary

This cell first checks that the experiment generated all required files. It will stop with a clear instruction if Step 2 was not completed successfully.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd

required_outputs = [Path('results/experiment_results.csv'), Path('results/summary_metrics.csv')]
missing = [str(p) for p in required_outputs if not p.is_file()]
if missing:
    raise FileNotFoundError('Cannot plot yet. Run Step 2 successfully first. Missing: ' + ', '.join(missing))

subprocess.run([sys.executable, 'scripts/plot_from_csv.py'], check=True)
summary = pd.read_csv('results/summary_metrics.csv')
display(summary)
print('\nFiles in results/:')
for path in sorted(Path('results').glob('*')):
    print(path.name, '-', path.stat().st_size, 'bytes')

## 4. Optional extended run (405 records)

Do this only after reviewing the pilot and confirming that the runtime has enough time and GPU resources. This run replaces the main result CSVs. The cell below first saves a copy of the pilot outputs so they are not lost.

In [ ]:
# OPTIONAL: uncomment and run this cell only when you are ready for the extended experiment.
# from pathlib import Path
# import shutil, subprocess, sys
# results_dir = Path('results')
# backup_dir = Path('results_pilot_backup')
# backup_dir.mkdir(exist_ok=True)
# for name in ['experiment_results.csv', 'summary_metrics.csv', 'run_metadata.json']:
#     source = results_dir / name
#     if source.exists():
#         shutil.copy2(source, backup_dir / name)
# subprocess.run([sys.executable, 'scripts/run_factorial_experiment.py', '--clusters', '15'], check=True)
# subprocess.run([sys.executable, 'scripts/plot_from_csv.py'], check=True)
# print('Extended run complete. Pilot outputs, if present, are in results_pilot_backup/.')

## Troubleshooting

- **`experiment_results.csv` or `summary_metrics.csv` missing:** Step 2 did not finish successfully. Read the error in Step 2; do not try to fix this by running the plotting cell again.
- **CUDA/GPU unavailable:** choose **Runtime → Change runtime type → GPU**, save, then rerun from Step 0.
- **Runtime disconnected or reset:** rerun the notebook from Step 0. Colab virtual machines can be reclaimed after inactivity or when runtime limits are reached.
- **Model/dataset download or memory error:** copy the complete first traceback from Step 2 and share it for diagnosis. Do not report incomplete runs as experimental results.
- **Reproducibility:** retain `run_metadata.json`, the raw `experiment_results.csv`, and the generated figures with the final paper. Clearly label the five-cluster pilot separately from the 15-cluster extended run.